# Multi-*e*GO — production simulation of the example systems

This notebook runs a complete **production multi-eGO simulation** for one of the example
systems distributed with the multi-eGO repository.  All pre-computed contact matrices and
reference topologies are already bundled in `tests/test_inputs/`, so the steps are:

| Step | Tool | What it does |
|------|------|--------------|
| 2 | `mego --config` | Generate the production force-field files from the bundled inputs |
| 3 | `gmx editconf` | Define the periodic simulation box |
| 4 | `gmx mdrun` | Energy minimisation |
| 5 | `gmx mdrun` | NVT production run |

> **Before you start:** go to *Runtime → Change runtime type* and select **GPU** to accelerate
> the production run.

---

## Choose your GROMACS installation (run **one** of the two options below)

| | Option A — apt | Option B — compile |
|-|---------------|-----------------|
| Time | ~2 min | ~20 min |
| Version | Ubuntu-packaged | release 2023 (recommended) |
| GPU support | depends on runtime | yes (CUDA auto-detected) |
| cmdata support | ✗ | ✓ |

Run **either** the *Option A* cell **or** the *Option B* cell — not both.

## ▶ Option A — Install GROMACS via apt  *(fast, ~2 min)*

Installs the GROMACS version packaged by Ubuntu.  Skip this cell and run
**Option B** instead if you want the multi-eGO GROMACS fork.

In [ ]:
# ── Option A — run this cell OR the Option B cell below, not both ────────────
!apt update -qq && apt install -y -qq gromacs
!gmx --version | head -5

## ▶ Option B — Compile GROMACS 2023  *(~20 min, recommended)*

Builds GROMACS with CUDA GPU support
when a GPU runtime is selected. FFTW3 is installed from apt
**This option is required if you also want to build `cmdata`.**

Skip this cell if you already ran **Option A** above.

In [ ]:
# ── Option B — run this cell OR the Option A cell above, not both ────────────
import subprocess, os

!apt update -qq && apt install -y -qq cmake build-essential libfftw3-dev ninja-build

# Clone the multi-eGO GROMACS fork (release-2023 branch)
!git clone --depth 1 -b release-2023 https://github.com/multi-ego/gromacs.git gromacs_src 2>/dev/null

# Configure
# - GMX_FFT_LIBRARY=fftw3   : use the system FFTW3 installed above
# - GMX_INSTALL_LEGACY_API  : required to build cmdata against this installation
# - CUDA is picked up automatically if a GPU runtime is active
!cmake -S gromacs_src -B gromacs_src/build -G Ninja \
       -DBUILD_TESTING=OFF \
       -DGMX_FFT_LIBRARY=fftw3 \
       -DGMX_INSTALL_LEGACY_API=ON \
       -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs_mego \
       > /dev/null 2>&1

# Build and install
!ninja -C gromacs_src/build -j$(nproc) 2>/dev/null
!ninja -C gromacs_src/build install 2>/dev/null

subprocess.run(["ln", "-sf", "/usr/local/gromacs_mego/bin/gmx", "/usr/local/bin/gmx"], check=True)
!gmx --version | head -5

## 1 · Install multi-eGO

In [ ]:
import sys, subprocess, os

# ── Pin pandas to <3.0.0 to avoid breaking Colab's pre-installed packages ─────
import pandas as _pd
_cur_pd = tuple(int(x) for x in _pd.__version__.split(".")[:3])
if _cur_pd < (2, 2, 3) or _cur_pd >= (3, 0, 0):
    print(f"Pinning pandas {_pd.__version__} → >=2.2.3,<3.0.0 …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2.3,<3.0.0"])
else:
    print(f"pandas {_pd.__version__} ✓")

!git clone --depth 1 https://github.com/multi-ego/multi-ego.git > /dev/null 2>&1
!pip install -q -e multi-ego

os.environ["GMXLIB"] = os.path.abspath("multi-ego")
print("GMXLIB =", os.environ["GMXLIB"])

## 2 · Select an example system

Choose one of the systems below, then run the cell.  All required contact matrices
and reference topologies are already bundled in the repository — no prior mg
simulation is needed.

| Option | System | Complexity |
|--------|--------|------------|
| GB1 protein | 56-residue β-sheet protein | single chain, intra contacts |
| Aβ42 peptide | 42-residue amyloid-forming peptide | single chain, intra contacts |
| TTR — peptide | 11 residue from transthyretin | trained from monomer and fibril |
| Lysozyme + benzene | 129-residue protein with small molecule | two-component system |

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import os

MEGO_ROOT  = os.path.abspath("multi-ego")
TEST_IN    = os.path.join(MEGO_ROOT, "tests", "test_inputs")

SYSTEMS = {
    "GB1 protein (gpref)": {
        "system": "gpref",
        "config": f"{TEST_IN}/gpref/config.yml",
        "gro":    f"{TEST_IN}/gpref/gb1_mego.gro",
    },
    "Aβ42 peptide (abetaref)": {
        "system": "abetaref",
        "config": f"{TEST_IN}/abetaref/config.yml",
        "gro":    f"{TEST_IN}/abetaref/ab42_mego.gro",
    },
    "TTR peptide — combined contacts (ttrref, config-3)": {
        "system": "ttrref",
        "config": f"{TEST_IN}/ttrref/config-3.yml",
        "gro":    f"{TEST_IN}/ttrref/ttr_mego.gro",
    },
    "Lysozyme + benzene (lyso-bnz_ref)": {
        "system": "lyso-bnz_ref",
        "config": f"{TEST_IN}/lyso-bnz_ref/config.yml",
        "gro":    f"{TEST_IN}/lyso-bnz_ref/lyso_bnz_mego.gro",
    },
}

selector = widgets.Dropdown(
    options=list(SYSTEMS.keys()),
    description="Example:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="550px"),
)
display(selector)

In [ ]:
cfg    = SYSTEMS[selector.value]
SYSTEM = cfg["system"]
CONFIG = cfg["config"]
GRO    = cfg["gro"]

print(f"System : {selector.value}")
print(f"Config : {CONFIG}")
print(f"GRO    : {GRO}")

## 2 · Generate the production force field  (`mego --config`)

Reads the pre-computed contact matrices and reference topology bundled in the repository
and writes the production force-field files:

- `topol_mego.top` — GROMACS topology with multi-eGO interactions
- `ffnonbonded.itp` — C6/C12 non-bonded parameters

In [ ]:
import subprocess, shutil, os, glob

OUTPUTS_DIR = os.path.abspath("mego_outputs")

result = subprocess.run(
    ["mego", "--config", CONFIG, "--outputs_dir", OUTPUTS_DIR],
    capture_output=True, text=True,
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("mego failed — see output above")

# Locate the output subdirectory (mego appends _1, _2, … automatically)
out_dirs = sorted(glob.glob(os.path.join(OUTPUTS_DIR, SYSTEM, "production_*")))
if not out_dirs:
    raise RuntimeError(f"mego output not found under {OUTPUTS_DIR}/{SYSTEM}/production_*/")
output_dir = out_dirs[-1]
print(f"\nmego output: {output_dir}")

# Copy force-field files to working directory so grompp can find them
shutil.copy(os.path.join(output_dir, "topol_mego.top"), "topol_mego.top")
shutil.copy(os.path.join(output_dir, "ffnonbonded.itp"), "ffnonbonded.itp")
print("Files ready: topol_mego.top  ffnonbonded.itp ✓")

## 3 · Set the simulation box  (`gmx editconf`)

Creates a cubic periodic box around the bundled reference structure.
The `-d` padding (in nm) ensures the protein does not interact with its periodic image.

In [ ]:
BOX_PADDING = 5.0  # nm

!gmx editconf -f {GRO} -o boxed.gro -c -bt cubic -d {BOX_PADDING} 2>&1 | tail -8

## 4 · Simulation parameters

Edit the values below before running the cell.  MDP files for energy minimisation
and the NVT production run are written from the multi-eGO templates.

In [ ]:
# ── Adjust these values ────────────────────────────────────────────────────────
TEMPERATURE_K = 300      # simulation temperature (K)
NSTEPS        = 1_000_000  # production steps  (1 M × 5 fs = 5 ns)
# ───────────────────────────────────────────────────────────────────────────────

def fill_mdp(template_path, output_path, subs):
    with open(template_path) as f:
        content = f.read()
    for key, val in subs.items():
        content = content.replace(key, str(val))
    with open(output_path, "w") as f:
        f.write(content)

fill_mdp("multi-ego/mdps/ff_em.mdp",  "em.mdp",     {})
fill_mdp("multi-ego/mdps/ff_aa.mdp",  "ff_aa.mdp",  {"_TEMP_": TEMPERATURE_K, "_SET_": NSTEPS})

print(f"em.mdp    → steepest-descent energy minimisation")
print(f"ff_aa.mdp → {NSTEPS:,} steps at {TEMPERATURE_K} K  ({NSTEPS * 0.005 / 1000:.1f} ns)")

## 5 · Energy minimisation  (`gmx mdrun`)

Steepest-descent minimisation with the production multi-eGO force field.

In [ ]:
!gmx grompp -f em.mdp -c boxed.gro -p topol_mego.top -o em.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm em -ntmpi 1
print("\nMinimisation complete ✓  →  em.gro")

## 6 · NVT production run  (`gmx mdrun`)

Langevin dynamics with the production multi-eGO force field.  The run produces:

- `run.xtc` — trajectory
- `run.tpr` — run-input file
- `run.edr` — energy file

In [ ]:
!gmx grompp -f ff_aa.mdp -c em.gro -p topol_mego.top -o run.tpr -maxwarn 1 2>&1 | tail -5
!gmx mdrun -v -deffnm run -ntmpi 1
print("\nProduction run complete ✓  →  run.xtc  run.tpr  run.edr")

## 7 · Energy analysis

Extracts potential energy and temperature from the GROMACS energy file and plots them.
Both quantities should be stable by the end of the run.

In [ ]:
import subprocess
import numpy as np
import matplotlib.pyplot as plt

subprocess.run(
    "printf 'Potential\\nTemperature\\n0\\n' | gmx energy -f run.edr -o energy.xvg",
    shell=True, capture_output=True,
)

time_ns, potential, temperature = [], [], []
with open("energy.xvg") as f:
    for line in f:
        if line.startswith(("#", "@")):
            continue
        cols = line.split()
        if len(cols) >= 3:
            time_ns.append(float(cols[0]) / 1000)
            potential.append(float(cols[1]))
            temperature.append(float(cols[2]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(time_ns, potential, color="steelblue", lw=0.8)
ax1.set_xlabel("Time (ns)")
ax1.set_ylabel("Potential energy (kJ/mol)")
ax1.set_title("Potential Energy")
ax1.grid(alpha=0.3)

ax2.plot(time_ns, temperature, color="tomato", lw=0.8)
ax2.axhline(TEMPERATURE_K, color="k", ls="--", lw=1, label=f"{TEMPERATURE_K} K target")
ax2.set_xlabel("Time (ns)")
ax2.set_ylabel("Temperature (K)")
ax2.set_title("Temperature")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("energy_plot.png", dpi=150)
plt.show()

print(f"Mean temperature : {np.mean(temperature):.1f} ± {np.std(temperature):.1f} K")
print(f"Mean pot. energy : {np.mean(potential):.1f} kJ/mol")

## 8 · Download output files

| File | Description |
|------|-------------|
| `run.xtc` | trajectory |
| `run.tpr` | run-input file |
| `topol_mego.top` | production multi-eGO topology |
| `ffnonbonded.itp` | non-bonded parameters |
| `energy_plot.png` | energy / temperature plot |

In [ ]:
from google.colab import files as colab_files
import os

for f in ["run.xtc", "run.tpr", "topol_mego.top", "ffnonbonded.itp", "energy_plot.png"]:
    if os.path.exists(f):
        print(f"Downloading {f}…")
        colab_files.download(f)
    else:
        print(f"  (skipping {f} — not found)")